# 추가실험 결과 표 (실험1 · 실험2)

4노드 eval(`exp_eval_node1~4`)이 끝난 뒤 실행. **상태 → SR 표 → 떨림 표 → zip**.
`eval_final` 과 동일한 계산식(SR: pooled Wilson CI / 떨림: aloha smooth_metrics_paper).

- **실험1** = ACT·BiMamba 각각 {없음, +TE(기존 smoothing), +MOSAIC(overlap)} — overlap이 TE보다, carry 있는 쪽이 없는 쪽보다 매끄러운지.
- **실험2** = MOSAIC = carry ⊕ overlap 분해 (acm2 / +carry / +overlap / +둘다).


## 0) 부팅 + 모델 정의


In [ ]:
import sys, json, csv, time
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)
import numpy as np
import smooth_metrics_paper as smp        # 팀원(aloha)과 동일한 스무스니스 계산식
importlib.reload(smp)

TASK   = 'libero_10'
SEEDS  = [0, 1]
TARGET_EP = 100                          # 추가실험 = task당 100ep. LIBERO-10 = 10 task → overall 1000
N_TASKS = 10
MIN_VALID_EP = N_TASKS * TARGET_EP // 2   # overall n_ep >= 500 만 유효(미완=빈칸)
MIN_TRAJ = 80                             # 궤적 이보다 적으면 미완/잔재로 보고 제외 (100ep→최대 1000)
FS     = cf.fps_of(TASK)                  # libero=30 (SPARC 주파수축)
STRIDE = 100                              # 청크 경계(하드스위치) stride, aloha 동일

# 실험별 (folder_tag, 표기 라벨, 역할)
EXP1 = [('act','ACT','baseline'), ('act_te','ACT + TE','baseline'),
        ('act_overlap','ACT + MOSAIC (overlap only)','abl'),
        ('bimamba','BiMamba','baseline'), ('bimamba_te','BiMamba + TE','baseline'),
        ('bimamba_mosaic','BiMOS (ours)','full')]
EXP2 = [('acm2','ACM2 (base)','baseline'), ('acm2_carry','ACM2 + carry','abl'),
        ('acm2_overlap','ACM2 + overlap','abl'), ('acm2_mosaic','ACM2 + MOSAIC (carry+overlap)','full')]
for t, _, _ in EXP1 + EXP2:
    v23.MODEL_DIR_NAMES.setdefault(t, t)

EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
OUT = cf.OUTPUT_BASE / 'share' / 'exp_results'
OUT.mkdir(parents=True, exist_ok=True)
STAMP = time.strftime('%Y%m%d_%H%M')

def eval_rec(tag, seed):                  # 그 tag×seed 의 최고 n_ep eval 기록
    d = EVAL_ROOT / tag / f'seed{seed}'
    if not d.is_dir(): return None
    best = None
    for info in d.rglob('eval_info.json'):
        try: ov = json.loads(info.read_text()).get('overall', {})
        except Exception: continue
        n_ep = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or n_ep > best['n_ep']:
            best = {'sr': ov.get('pc_success'), 'n_ep': n_ep,
                    'has_actions': (info.parent / 'actions').is_dir(), 'path': info.parent}
    return best

def save_table_png(col_labels, cell_text, title, path):
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    nrow, ncol = len(cell_text), len(col_labels)
    fig, ax = plt.subplots(figsize=(1.4 + 1.25 * ncol, 0.6 + 0.42 * (nrow + 1)))
    ax.axis('off')
    tb = ax.table(cellText=cell_text, colLabels=col_labels, loc='center', cellLoc='center')
    tb.auto_set_font_size(False); tb.set_fontsize(10)
    tb.auto_set_column_width(col=list(range(ncol))); tb.scale(1, 1.5)
    for (r, c), cell in tb.get_celld().items():
        if r == 0: cell.set_text_props(weight='bold'); cell.set_facecolor('#e8e8e8')
        if c == 0 and r > 0: cell.set_text_props(ha='left'); cell.PAD = 0.04
    ax.set_title(title, fontsize=12, pad=10)
    fig.savefig(path, dpi=150, bbox_inches='tight'); plt.close(fig)
    print('  이미지:', path)

print('eval:', EVAL_ROOT); print('저장:', OUT, '| 스냅샷', STAMP)

## 1) 상태 (어떤 셀이 채워질지)


In [ ]:
# 각 실험 모델×seed: eval 됐나 / n_ep / action 있나 (재학습 없으니 '학습' 열은 생략)
def status_block(desired, name):
    print(f'━━ {name} ━━')
    print(f'{"model":<32}{"seed":>5}{"eval":>10}{"action":>8}')
    for tag, lb, _ in desired:
        for s in SEEDS:
            ev = eval_rec(tag, s)
            ok = ev and (ev['n_ep'] or 0) >= MIN_VALID_EP
            ev_s = f"{ev['n_ep']}ep" if ev else '-'
            act = 'O' if (ev and ev['has_actions']) else '-'
            print(f'{tag+" ("+lb+")":<32}{s:>5}{ev_s:>10}{act:>8}')
    print()
status_block(EXP1, '실험1: ACT vs BiMamba, MOSAIC vs TE')
status_block(EXP2, '실험2: MOSAIC 구성요소 분해')

## 2) SR 표 (seed별 · mean±std · pooled 95% CI)


In [ ]:
# SR 표: 유효 500ep 만 · seed별 · mean±std · pooled 95% CI. CSV+PNG 저장.
def sr_table(desired, name):
    hdr = f'{"model":<32}' + ''.join(f'{("s"+str(s)):>7}' for s in SEEDS) + f'{"mean":>8}{"±std":>7}{"pooled95%CI":>16}'
    print(f'━━ {name} ━━'); print(hdr); print('-' * len(hdr))
    rows = []
    for tag, lb, role in desired:
        per, pk, pn = {}, 0, 0
        for s in SEEDS:
            ev = eval_rec(tag, s)
            if ev and (ev['n_ep'] or 0) >= MIN_VALID_EP and ev['sr'] is not None:
                per[s] = ev['sr']; n = int(ev['n_ep']); pk += int(round(ev['sr'] / 100 * n)); pn += n
        vals = list(per.values())
        mean = float(np.mean(vals)) if vals else None
        std = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if vals else None)
        ci = ''
        if pn:
            lo, hi = v23.wilson_ci(pk, pn); ci = f'[{lo*100:.1f},{hi*100:.1f}]'
        cells = ''.join((f'{per[s]:>7.1f}' if s in per else f'{chr(32)*7}') for s in SEEDS)
        mean_s = f'{mean:>8.1f}' if mean is not None else f'{chr(32)*8}'
        std_s = f'{std:>7.1f}' if std is not None else f'{chr(32)*7}'
        print(f'{lb:<32}{cells}{mean_s}{std_s}{ci:>16}')
        rows.append({'model': tag, 'label': lb, 'role': role,
                     **{f'seed{s}': (round(per[s], 1) if s in per else None) for s in SEEDS},
                     'mean': (round(mean, 2) if mean is not None else None),
                     'std': (round(std, 2) if std is not None else None), 'n_seed': len(vals),
                     'pooled_sr': (round(pk / pn * 100, 2) if pn else None), 'pooled_ci': ci})
    cols = ['model', 'label', 'role'] + [f'seed{s}' for s in SEEDS] + ['mean', 'std', 'n_seed', 'pooled_sr', 'pooled_ci']
    with open(OUT / f'sr_{name}_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=cols); w.writeheader(); w.writerows(rows)
    img_col = ['model'] + [f's{s}' for s in SEEDS] + ['mean', '±std', 'n', 'pooled 95%CI']
    img_txt = [[r['label']] + [(f"{r['seed'+str(s)]:.1f}" if r['seed'+str(s)] is not None else '') for s in SEEDS]
               + ['' if r['mean'] is None else f"{r['mean']:.1f}", '' if r['std'] is None else f"{r['std']:.1f}",
                  (f"{r['n_seed']}/{len(SEEDS)}" if r['n_seed'] else ''), r['pooled_ci'] or ''] for r in rows]
    save_table_png(img_col, img_txt, f'{name} — SR ({TARGET_EP}ep)  {STAMP}', OUT / f'sr_{name}_{STAMP}.png')
    print()
sr_table(EXP1, 'exp1')
sr_table(EXP2, 'exp2')

## 3) 떨림 표 (aloha 방식 5지표)


In [ ]:
# 떨림 표 (aloha 방식): 경계/내부 jerk RMS · B/I · SPARC · ldj_cost · signflip. CSV+PNG.
def smooth_table(desired, name):
    print(f'━━ {name} ━━')
    print(f'{"model":<30}{"jerk":>9}{"bnd":>9}{"int":>9}{"B/I":>7}{"SPARC":>9}{"ldj":>9}{"sflip":>9}{"n":>6}')
    print('  방향:            ↓        ↓        ↓     →1     →0        ↓        ↓')
    rows = []
    for tag, lb, role in desired:
        trajs = []
        for s in SEEDS:
            ev = eval_rec(tag, s)
            if not (ev and (ev['n_ep'] or 0) >= MIN_VALID_EP and ev['has_actions']): continue
            tr = v23._load_action_trajs(ev['path'] / 'actions') or []
            if len(tr) >= MIN_TRAJ: trajs += tr
        a = smp.aggregate_paper(trajs, boundary_stride=STRIDE, fs=FS) if trajs else None
        if a:
            print(f'{lb:<30}{a["jerk_rms_mean"]:>9.4f}{a["boundary_jerk_rms_mean"]:>9.4f}'
                  f'{a["interior_jerk_rms_mean"]:>9.4f}{a["boundary_interior_ratio_mean"]:>7.2f}'
                  f'{a["sparc_mean"]:>9.2f}{a["ldj_cost_mean"]:>9.2f}{a["sign_flip_rate_mean"]:>9.4f}{a["n_traj"]:>6}')
            rows.append({'model': tag, 'label': lb, 'jerk_rms': round(a['jerk_rms_mean'], 5),
                         'boundary_jerk_rms': round(a['boundary_jerk_rms_mean'], 5),
                         'interior_jerk_rms': round(a['interior_jerk_rms_mean'], 5),
                         'boundary_interior_ratio': round(a['boundary_interior_ratio_mean'], 4),
                         'sparc': round(a['sparc_mean'], 3), 'ldj_cost': round(a['ldj_cost_mean'], 3),
                         'sign_flip_rate': round(a['sign_flip_rate_mean'], 5), 'n_traj': a['n_traj']})
        else:
            print(f'{lb:<30}' + ' ' * 61 + '(빈칸)')
            rows.append({'model': tag, 'label': lb, 'jerk_rms': None, 'boundary_jerk_rms': None,
                         'interior_jerk_rms': None, 'boundary_interior_ratio': None, 'sparc': None,
                         'ldj_cost': None, 'sign_flip_rate': None, 'n_traj': 0})
    with open(OUT / f'smooth_{name}_{STAMP}.csv', 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['model', 'label', 'jerk_rms', 'boundary_jerk_rms',
            'interior_jerk_rms', 'boundary_interior_ratio', 'sparc', 'ldj_cost', 'sign_flip_rate', 'n_traj'])
        w.writeheader(); w.writerows(rows)
    def _f(v, d): return '' if v is None else f'{v:.{d}f}'
    img_col = ['model', 'jerk↓', 'bnd↓', 'int↓', 'B/I→1', 'SPARC→0', 'ldj↓', 'sflip↓', 'n']
    img_txt = [[r['label'], _f(r['jerk_rms'], 4), _f(r['boundary_jerk_rms'], 4), _f(r['interior_jerk_rms'], 4),
                _f(r['boundary_interior_ratio'], 2), _f(r['sparc'], 2), _f(r['ldj_cost'], 2),
                _f(r['sign_flip_rate'], 4), (str(r['n_traj']) if r['n_traj'] else '')] for r in rows]
    save_table_png(img_col, img_txt, f'{name} — smoothness (aloha, fs={FS})  {STAMP}', OUT / f'smooth_{name}_{STAMP}.png')
    print()
smooth_table(EXP1, 'exp1')
smooth_table(EXP2, 'exp2')

## 4) 요약 + zip


In [ ]:
# 요약 + zip (SR/떨림 표 CSV·PNG 는 위에서 저장됨)
import shutil
(OUT / f'README_{STAMP}.md').write_text(
    f'# 추가실험 결과 (LIBERO-10, {TARGET_EP}ep, {STAMP})\n\n'
    f'- 실험1 = ACT/BiMamba × (없음/TE/MOSAIC), 실험2 = MOSAIC 구성요소(carry⊕overlap) 분해\n'
    f'- 떨림 = aloha 동일 스크립트(smooth_metrics_paper), fs={FS}, 경계 stride={STRIDE}\n'
    f'- 전부 재학습 없이 기존 ckpt 재해석/eval-time 적용\n', encoding='utf-8')
zip_path = shutil.make_archive(str(cf.OUTPUT_BASE / 'share' / f'exp_results_{STAMP}'), 'zip', root_dir=OUT)
print('보낼 파일:', zip_path)
for p in sorted(OUT.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(OUT)}  ({p.stat().st_size/1024:.0f} KB)')